In [14]:
%%writefile products.csv
itemName,Category,units,Amount
Phone,Electronics,5,60000
PowerBank,Utility,20,900
Laptop,Electronics,10,150000
Headphones,Electronics,15,7500
Chair,Furniture,30,45000
Table,Furniture,25,60000
Notebook,Stationery,50,2500
Pen,Stationery,100,1500
Bag,Accessories,10,5000
Watch,Accessories,7,35000
Shirt,Apparel,40,8000
Shoes,Apparel,30,18000
Bottle,Utility,50,5000
Speakers,Electronics,20,30000
Mouse,Electronics,35,3500
Book,Stationery,60,6000
Fan,Appliances,12,24000
Lamp,Furniture,20,20000
Glasses,Accessories,15,22500

Overwriting products.csv


In [15]:
#Import Libraries
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [16]:
#Create SparkSession with name "Superstore Data Analysis"
spark=SparkSession.builder.appName("Superstore Data Analysis").getOrCreate()

In [17]:
#Create Dataframe from data using path variable
df=spark.read.csv("products.csv",header=True,inferSchema=True)
df.show()

+----------+-----------+-----+------+
|  itemName|   Category|units|Amount|
+----------+-----------+-----+------+
|     Phone|Electronics|    5| 60000|
| PowerBank|    Utility|   20|   900|
|    Laptop|Electronics|   10|150000|
|Headphones|Electronics|   15|  7500|
|     Chair|  Furniture|   30| 45000|
|     Table|  Furniture|   25| 60000|
|  Notebook| Stationery|   50|  2500|
|       Pen| Stationery|  100|  1500|
|       Bag|Accessories|   10|  5000|
|     Watch|Accessories|    7| 35000|
|     Shirt|    Apparel|   40|  8000|
|     Shoes|    Apparel|   30| 18000|
|    Bottle|    Utility|   50|  5000|
|  Speakers|Electronics|   20| 30000|
|     Mouse|Electronics|   35|  3500|
|      Book| Stationery|   60|  6000|
|       Fan| Appliances|   12| 24000|
|      Lamp|  Furniture|   20| 20000|
|   Glasses|Accessories|   15| 22500|
+----------+-----------+-----+------+



In [18]:
#Add a new column TotalAmount (this will be product of quantity & amount)
df=df.withColumn("TotalAmount", F.col("units")*F.col("Amount"))
df.show()

#Further add a DiscountedAmount column which has 10% discount applied
df=df.withColumn("DiscountedAmount", F.col("TotalAmount")*0.9)
df.show()


+----------+-----------+-----+------+-----------+
|  itemName|   Category|units|Amount|TotalAmount|
+----------+-----------+-----+------+-----------+
|     Phone|Electronics|    5| 60000|     300000|
| PowerBank|    Utility|   20|   900|      18000|
|    Laptop|Electronics|   10|150000|    1500000|
|Headphones|Electronics|   15|  7500|     112500|
|     Chair|  Furniture|   30| 45000|    1350000|
|     Table|  Furniture|   25| 60000|    1500000|
|  Notebook| Stationery|   50|  2500|     125000|
|       Pen| Stationery|  100|  1500|     150000|
|       Bag|Accessories|   10|  5000|      50000|
|     Watch|Accessories|    7| 35000|     245000|
|     Shirt|    Apparel|   40|  8000|     320000|
|     Shoes|    Apparel|   30| 18000|     540000|
|    Bottle|    Utility|   50|  5000|     250000|
|  Speakers|Electronics|   20| 30000|     600000|
|     Mouse|Electronics|   35|  3500|     122500|
|      Book| Stationery|   60|  6000|     360000|
|       Fan| Appliances|   12| 24000|     288000|


In [19]:
#Filter items with category "Furniture"
df.filter(F.col("Category")=="Furniture").show()

+--------+---------+-----+------+-----------+----------------+
|itemName| Category|units|Amount|TotalAmount|DiscountedAmount|
+--------+---------+-----+------+-----------+----------------+
|   Chair|Furniture|   30| 45000|    1350000|       1215000.0|
|   Table|Furniture|   25| 60000|    1500000|       1350000.0|
|    Lamp|Furniture|   20| 20000|     400000|        360000.0|
+--------+---------+-----+------+-----------+----------------+



In [20]:
#Create an SQL view named "superstoredf"
superstoredf=df.createOrReplaceTempView("superstoredf")

#Create an SQL view named "supermartdata"
supermartdata=df.createOrReplaceTempView("supermartdata")


In [31]:
#Calculate totalAmount amount using column "Amount"
df1=spark.sql("""
Select sum(Amount) from superstoredf
""")
df1.show()

#Calculate average amount per category
df2=spark.sql("""
Select Category, round(avg(Amount),2) as average from superstoredf
group by Category
""")
df2.show()

#Find highest total price product
df2=spark.sql("""
Select itemName, TotalAmount from superstoredf
order by TotalAmount desc
limit 1
""")
df2.show()


+-----------+
|sum(Amount)|
+-----------+
|     504400|
+-----------+

+-----------+--------+
|   Category| average|
+-----------+--------+
| Stationery| 3333.33|
|    Apparel| 13000.0|
|Electronics| 50200.0|
|    Utility|  2950.0|
|  Furniture|41666.67|
|Accessories|20833.33|
| Appliances| 24000.0|
+-----------+--------+

+--------+-----------+
|itemName|TotalAmount|
+--------+-----------+
|  Laptop|    1500000|
+--------+-----------+



In [ ]:
#Find number of products available per category (Group by Category & count itemName)

